# Diagnosis to Medication

## Architectural Flow

![Architectural Flow](architecture_flow.png)

# Diagnosis to Medication Validation

This notebook validates whether a patient's medications are associated with a given diagnosis
according to the IMO Health Knowledge Graph. For a diagnosis, the KG returns **medication groups**
(e.g., "ACE Inhibitors", "Beta Blockers") and the specific medications within each group.
We then check whether each of the patient's medications appears in any group.

## Tool chain used in `dx-to-med` mode
```python
DX_TO_MED_TOOLS = [
    normalize_medical_term,    # Normalize diagnosis & medications
    get_medication_groups,     # Query KG for medication groups
]
```

## Key implementation details
- Diagnosis is normalized as `domain="Problem"` to get its `default_lexical_code`
- `get_medication_groups` queries KG for `medicationGroups` on ProblemLexical
- Each patient medication is normalized as `domain="Medication"` to get its code
- Match is **code-based only** — medication code must appear in a KG group's medication list
- No title/name matching — only exact IMO code equality counts

## Workflow

| Step | Action | API |
|------|--------|-----|
| 1 | Normalize diagnosis (domain=Problem) | IMO Normalize |
| 2 | Query medication groups from KG | IMO KG GraphQL |
| 3 | Normalize each patient medication (domain=Medication) | IMO Normalize |
| 4 | Match medication codes against KG groups | — |
| 5 | Report associated vs. not-associated medications | — |

## 0. Setup — paths, imports, credentials

In [ ]:
import sys
import os

# Ensure the notebook's own directory is on the path so local
# kg_config.py and kg_api_client.py are imported directly.
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

import json
from IPython.display import display, Markdown

# ── Credential check ────────────────────────────────────────────────────────
# kg_config.py reads credentials from AWS SSM or environment variables.
# If SSM is unavailable locally, uncomment and set the env vars below.
#
# os.environ["IMO_NORMALIZE_CLIENT_ID"] = "..."
# os.environ["IMO_NORMALIZE_SECRET"]    = "..."
# os.environ["IMO_KG_CLIENT_ID"]        = "..."
# os.environ["IMO_KG_CLIENT_SECRET"]    = "..."

import kg_config as kc

missing = []
if not kc.IMO_NORMALIZE_CLIENT_ID:     missing.append("IMO_NORMALIZE_CLIENT_ID")
if not kc.IMO_NORMALIZE_CLIENT_SECRET: missing.append("IMO_NORMALIZE_SECRET")
if not kc.IMO_KG_CLIENT_ID:           missing.append("IMO_KG_CLIENT_ID")
if not kc.IMO_KG_CLIENT_SECRET:       missing.append("IMO_KG_CLIENT_SECRET")

if missing:
    print(f"\u26a0  Missing credentials: {missing}")
    print("   Set them as env vars (see comment above) or ensure AWS SSM access.")
else:
    print("\u2713  Credentials loaded")
    print(f"   Normalize URL : {kc.IMO_NORMALIZE_URL}")
    print(f"   KG GraphQL URL: {kc.KG_GRAPHQL_URL}")

## 1. Clinical Scenario

A 62-year-old patient with a diagnosis of **Hypertension** is on the following
medication regimen. We want to validate which medications are associated with
hypertension per the IMO Knowledge Graph and which are not.

**Diagnosis:** Hypertension

**Patient medications:**
- Lisinopril
- Metformin
- Amlodipine
- Atorvastatin
- Metoprolol

In [ ]:
DIAGNOSIS = "Hypertension"

PATIENT_MEDICATIONS = [
    "Lisinopril",
    "Metformin",
    "Amlodipine",
    "Atorvastatin",
    "Metoprolol",
]

print(f"Diagnosis  : {DIAGNOSIS}")
print(f"Medications: {PATIENT_MEDICATIONS}")

## 2. Initialize KGApiClient

In [ ]:
from kg_api_client import KGApiClient

client = KGApiClient()
print("KGApiClient ready")

## Step 1 — Normalize the Diagnosis

Calls `normalize_medical_term(input_term, domain="Problem")` for the diagnosis.
Returns a `default_lexical_code` used to query medication groups from the KG.

In [ ]:
print(f"Normalizing diagnosis (domain=Problem): '{DIAGNOSIS}'...\n")

dx_result = client.normalize_medical_term(DIAGNOSIS, "Problem")
diagnosis_code = None
diagnosis_title = None

if dx_result.get("success") and dx_result.get("results"):
    matches = dx_result["results"][0].get("matches", [])
    if matches:
        best = matches[0]
        diagnosis_code = best["default_lexical_code"]
        diagnosis_title = best["title"]
        print(f"  \u2713  '{DIAGNOSIS}'")
        print(f"       code  = {diagnosis_code}")
        print(f"       title = '{diagnosis_title}'")
        print(f"       score = {best['score']}")
        if best.get("icd10_codes"):
            print(f"       ICD-10: {best['icd10_codes']}")
    else:
        print(f"  \u2717  No matches returned for '{DIAGNOSIS}'")
else:
    print(f"  \u2717  API error: {dx_result.get('error')}")

## Step 2 — Query Medication Groups from KG

Calls `get_medication_groups(imo_lexical_code)` on the normalized diagnosis code.
Returns a list of medication groups, each with a group title and the medications in it.

In [ ]:
print(f"Querying medication groups for '{diagnosis_title}' (code={diagnosis_code})...\n")

groups_result = client.get_medication_groups(diagnosis_code)

medication_groups = []
all_kg_med_codes = {}  # code -> {title, group_title, group_code}

if groups_result.get("success") and groups_result.get("lexical"):
    medication_groups = groups_result["lexical"].get("medicationGroups", []) or []
    print(f"  \u2713  Found {len(medication_groups)} medication groups\n")
    
    for group in medication_groups:
        meds_in_group = group.get("medications", []) or []
        print(f"  Group: '{group['title']}' (code={group['code']}) — {len(meds_in_group)} medications")
        for med in meds_in_group:
            all_kg_med_codes[med["code"]] = {
                "title": med["title"],
                "group_title": group["title"],
                "group_code": group["code"],
            }
        # Show first few medications in each group
        for med in meds_in_group[:5]:
            print(f"    - {med['title']} (code={med['code']})")
        if len(meds_in_group) > 5:
            print(f"    ... and {len(meds_in_group) - 5} more")
        print()
    
    print(f"  Total unique medication codes across all groups: {len(all_kg_med_codes)}")
else:
    print(f"  \u2717  Error: {groups_result.get('error', 'no data returned')}")

## Step 3 — Normalize Each Patient Medication & Match Against KG

For each medication in the patient's list:
1. Normalize it (`domain="Medication"`) to get its `default_lexical_code`
2. Check if that code appears in ANY medication group from Step 2
3. A match is **only valid by exact code** — not by title similarity

In [ ]:
associated = []     # {input, title, code, group_title, group_code}
not_associated = [] # {input, title, code, reason}

print("Normalizing patient medications and matching against KG groups...\n")

for med_name in PATIENT_MEDICATIONS:
    result = client.normalize_medical_term(med_name, "Medication")
    
    if not result.get("success") or not result.get("results"):
        not_associated.append({
            "input": med_name,
            "title": "—",
            "code": "—",
            "reason": "Could not normalize — skipped",
        })
        print(f"  \u2717  {med_name:<20}  Could not normalize")
        continue
    
    matches = result["results"][0].get("matches", [])
    if not matches:
        not_associated.append({
            "input": med_name,
            "title": "—",
            "code": "—",
            "reason": "No normalize matches returned",
        })
        print(f"  \u2717  {med_name:<20}  No matches returned")
        continue
    
    best = matches[0]
    med_code = best["default_lexical_code"]
    med_title = best["title"]
    
    # Check if this code exists in any KG medication group
    if med_code in all_kg_med_codes:
        kg_info = all_kg_med_codes[med_code]
        associated.append({
            "input": med_name,
            "title": med_title,
            "code": med_code,
            "group_title": kg_info["group_title"],
            "group_code": kg_info["group_code"],
        })
        print(f"  \u2713  {med_name:<20}  code={med_code:<10}  \u2192 Group: '{kg_info['group_title']}'")
    else:
        not_associated.append({
            "input": med_name,
            "title": med_title,
            "code": med_code,
            "reason": "No match found in KG medicationGroups for this diagnosis",
        })
        print(f"  \u2717  {med_name:<20}  code={med_code:<10}  NOT in any KG group")

print(f"\n\u2713  {len(associated)}/{len(PATIENT_MEDICATIONS)} associated, {len(not_associated)} not associated")

## Step 4 — Validation Results

In [ ]:
lines = [
    "## Diagnosis-to-Medication Validation",
    "",
    f"**Diagnosis:** {diagnosis_title} (`{diagnosis_code}`)",
    "",
]

# Associated medications table
if associated:
    lines.append("**Associated Medications (found in KG):**")
    lines.append("")
    lines.append("| Medication (User Input) | Matched KG Title | Medication Code | Medication Group | Group Code |")
    lines.append("|---|---|---|---|---|")
    for item in associated:
        lines.append(f"| {item['input']} | {item['title']} | {item['code']} | {item['group_title']} | {item['group_code']} |")
    lines.append("")
else:
    lines.append("**Associated Medications:** None found in KG")
    lines.append("")

# Not associated medications table
if not_associated:
    lines.append("**Not Associated (not found in KG):**")
    lines.append("")
    lines.append("| Medication (User Input) | Normalized Title | IMO Code | Reason |")
    lines.append("|---|---|---|---|")
    for item in not_associated:
        lines.append(f"| {item['input']} | {item['title']} | {item['code']} | {item['reason']} |")
    lines.append("")

# Summary
lines.append("**Summary:**")
lines.append(f"- {len(associated)} of {len(PATIENT_MEDICATIONS)} medications matched to KG medication groups")
lines.append(f"- {len(not_associated)} medications not found in KG for this diagnosis")

display(Markdown("\n".join(lines)))

## Appendix A — Full Medication Groups from KG

Complete list of medication groups and their medications as returned by the KG
for this diagnosis.

In [ ]:
PREVIEW_N = 10

for group in medication_groups:
    meds = group.get("medications", []) or []
    print(f"{'='*60}")
    print(f"  Group: {group['title']} (code={group['code']})")
    print(f"  Medications: {len(meds)} total (showing first {min(PREVIEW_N, len(meds))})")
    print()
    for med in meds[:PREVIEW_N]:
        # Mark if this med was in our patient's list
        patient_match = ""
        for a in associated:
            if a["code"] == med["code"]:
                patient_match = f"  \u25c0 PATIENT MED: {a['input']}"
                break
        print(f"    code={med['code']:<10}  {med['title']}{patient_match}")
    if len(meds) > PREVIEW_N:
        print(f"    ... and {len(meds) - PREVIEW_N} more")
    print()

## Appendix B — Additional Clinical Scenarios

Try these alternative scenarios by changing `DIAGNOSIS` and `PATIENT_MEDICATIONS` above:

| Scenario | Diagnosis | Medications |
|----------|-----------|-------------|
| T2DM Validation | Type 2 diabetes mellitus | Metformin, Lisinopril, Insulin glargine, Atorvastatin, Empagliflozin |
| Heart Failure | Heart failure | Furosemide, Carvedilol, Lisinopril, Spironolactone, Amoxicillin |
| Asthma | Asthma | Albuterol, Fluticasone, Montelukast, Metformin, Budesonide |